In [4]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

# Download NLTK stop words data
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

print("1. Loading WELFake dataset...")
# Load dataset
df = pd.read_csv(r'c:\Users\chamo\OneDrive\Desktop\NLP_Group_09\WELFake_Dataset.csv')

print("2. Handling missing values and duplicates...")
# Handle missing text
df.dropna(subset=['text'], inplace=True)
df['title'] = df['title'].fillna('')

# Combine title and text into one single column
df['combined_text'] = df['title'] + " " + df['text']

# Remove duplicate entries
df.drop_duplicates(subset=['combined_text'], inplace=True)

print("3. Cleaning text (Lowercasing, removing URLs & punctuation)...")
def clean_text(text):
    text = text.lower()                                    # Lowercase
    text = re.sub(r'<.*?>', '', text)                      # Remove HTML tags
    text = re.sub(r'https?://\S+|www\.\S+', '', text)     # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)               # Keep only letters/spaces
    text = re.sub(r'\s+', ' ', text).strip()               # Remove extra spaces
    return text

df['clean_text'] = df['combined_text'].apply(clean_text)

# Path for SVM (Remove stop words)
print("4. Preparing clean text for SVM model...")
def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

df['svm_text'] = df['clean_text'].apply(remove_stopwords)

print("5. Saving processed data...")
# Save output to a new file
df.to_csv('cleaned_WELFake_Dataset.csv', index=False)

print("\n Done! Generated 'cleaned_WELFake_Dataset.csv'")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\chamo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


1. Loading WELFake dataset...
2. Handling missing values and duplicates...
3. Cleaning text (Lowercasing, removing URLs & punctuation)...
4. Preparing clean text for SVM model...
5. Saving processed data...

 Done! Generated 'cleaned_WELFake_Dataset.csv'


In [5]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download VADER lexicon for sentiment extraction
nltk.download('vader_lexicon')

print("1. Loading cleaned dataset...")
df = pd.read_csv('cleaned_WELFake_Dataset.csv')

print("2. Initializing VADER Sentiment Analyzer...")
sia = SentimentIntensityAnalyzer()

print("3. Extracting sentiment scores...")

def get_sentiment_scores(text):
    if not isinstance(text, str):
        return 0.0, 0.0, 0.0, 0.0
    scores = sia.polarity_scores(text)
    return scores['compound'], scores['pos'], scores['neu'], scores['neg']

sentiment_data = df['clean_text'].apply(get_sentiment_scores)

df['sentiment_compound'] = [s[0] for s in sentiment_data]
df['sentiment_pos'] = [s[1] for s in sentiment_data]
df['sentiment_neu'] = [s[2] for s in sentiment_data]
df['sentiment_neg'] = [s[3] for s in sentiment_data]

print("4. Sentiment Scores Sample:")
print(df[['clean_text', 'sentiment_compound', 'label']].head())

print("5. Saving dataset with sentiment features...")
df.to_csv('cleaned_WELFake_with_sentiment.csv', index=False)

print("\n Done! Output saved as 'cleaned_WELFake_with_sentiment.csv'")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\chamo\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


1. Loading cleaned dataset...
2. Initializing VADER Sentiment Analyzer...
3. Extracting sentiment scores...
4. Sentiment Scores Sample:
                                          clean_text  sentiment_compound  \
0  law enforcement on high alert following threat...             -0.9938   
1      did they post their votes for hillary already              0.0000   
2  unbelievable obamas attorney general says most...              0.8807   
3  bobby jindal raised hindu uses story of christ...              0.9993   
4  satan russia unvelis an image of its terrifyin...             -0.9442   

   label  
0      1  
1      1  
2      1  
3      0  
4      1  
5. Saving dataset with sentiment features...

 Done! Output saved as 'cleaned_WELFake_with_sentiment.csv'


In [6]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer

print("1. Loading dataset...")
df = pd.read_csv('cleaned_WELFake_with_sentiment.csv').dropna(subset=['clean_text'])

print("2. Initializing WordPiece BertTokenizer...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

print("3. Preparing DataLoader sample...")
dataset = FakeNewsDataset(df['clean_text'][:100], df['label'][:100], tokenizer)
loader = DataLoader(dataset, batch_size=16)

for batch in loader:
    print("Batch Input IDs shape:", batch['input_ids'].shape)
    print("Batch Attention Mask shape:", batch['attention_mask'].shape)
    print("Batch Labels shape:", batch['labels'].shape)
    break

print("\nWordPiece Tokenization Pipeline verified successfully!")

1. Loading dataset...
2. Initializing WordPiece BertTokenizer...
3. Preparing DataLoader sample...
Batch Input IDs shape: torch.Size([16, 128])
Batch Attention Mask shape: torch.Size([16, 128])
Batch Labels shape: torch.Size([16])

WordPiece Tokenization Pipeline verified successfully!


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from scipy.sparse import hstack

print("1. Loading dataset with sentiment features...")
df = pd.read_csv('cleaned_WELFake_with_sentiment.csv')

# Drop any potential NaN values in clean_text
df = df.dropna(subset=['clean_text'])

print("2. Splitting dataset into train and test sets (80/20)...")
X_text = df['clean_text']
X_sentiment = df[['sentiment_compound', 'sentiment_pos', 'sentiment_neu', 'sentiment_neg']].values
y = df['label']

X_text_train, X_text_test, X_sent_train, X_sent_test, y_train, y_test = train_test_split(
    X_text, X_sentiment, y, test_size=0.2, random_state=42, stratify=y
)

print("3. Extracting TF-IDF features...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_text_train)
X_test_tfidf = tfidf.transform(X_text_test)

print("4. Combining TF-IDF and Sentiment features...")
X_train_combined = hstack([X_train_tfidf, X_sent_train])
X_test_combined = hstack([X_test_tfidf, X_sent_test])

print("5. Training Support Vector Machine (LinearSVC)...")
svm_model = LinearSVC(max_iter=2000, random_state=42)
svm_model.fit(X_train_combined, y_train)

print("6. Evaluating Model Performance...\n")
y_pred = svm_model.predict(X_test_combined)

print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Real (0)', 'Fake (1)']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

1. Loading dataset with sentiment features...
2. Splitting dataset into train and test sets (80/20)...
3. Extracting TF-IDF features...
4. Combining TF-IDF and Sentiment features...
5. Training Support Vector Machine (LinearSVC)...
6. Evaluating Model Performance...

Accuracy: 96.05%

Classification Report:
              precision    recall  f1-score   support

    Real (0)       0.96      0.97      0.96      6959
    Fake (1)       0.96      0.95      0.96      5760

    accuracy                           0.96     12719
   macro avg       0.96      0.96      0.96     12719
weighted avg       0.96      0.96      0.96     12719

Confusion Matrix:
[[6722  237]
 [ 266 5494]]


In [8]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification 
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("1. Loading dataset...")
df = pd.read_csv('cleaned_WELFake_with_sentiment.csv').dropna(subset=['clean_text'])

# Split data into Train and Validation sets (using a sample for fast initial fine-tuning)
train_df, val_df = train_test_split(df.sample(n=5000, random_state=42), test_size=0.2, random_state=42)

print("2. Tokenizing data...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = FakeNewsDataset(train_df['clean_text'], train_df['label'], tokenizer)
val_dataset = FakeNewsDataset(val_df['clean_text'], val_df['label'], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

print("3. Initializing BERT Model...")
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

print("4. Starting Fine-Tuning (2 Epochs)...")
model.train()
for epoch in range(2):
    total_loss = 0
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

        if (step + 1) % 50 == 0:
            print(f"Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

print("\n5. Evaluating Fine-Tuned BERT Model...")
model.eval()
predictions, true_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()

        predictions.extend(preds)
        true_labels.extend(labels.numpy())

print(f"\nBERT Accuracy: {accuracy_score(true_labels, predictions) * 100:.2f}%\n")
print(classification_report(true_labels, predictions, target_names=['Real (0)', 'Fake (1)']))

Using device: cpu
1. Loading dataset...
2. Tokenizing data...


3. Initializing BERT Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


4. Starting Fine-Tuning (2 Epochs)...
Epoch 1 | Step 50/250 | Loss: 0.5116
Epoch 1 | Step 100/250 | Loss: 0.1448
Epoch 1 | Step 150/250 | Loss: 0.1423
Epoch 1 | Step 200/250 | Loss: 0.1089
Epoch 1 | Step 250/250 | Loss: 0.0376
Epoch 2 | Step 50/250 | Loss: 0.0409
Epoch 2 | Step 100/250 | Loss: 0.0465
Epoch 2 | Step 150/250 | Loss: 0.0118
Epoch 2 | Step 200/250 | Loss: 0.0070
Epoch 2 | Step 250/250 | Loss: 0.2756

5. Evaluating Fine-Tuned BERT Model...

BERT Accuracy: 96.70%

              precision    recall  f1-score   support

    Real (0)       0.99      0.95      0.97       560
    Fake (1)       0.94      0.99      0.96       440

    accuracy                           0.97      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.97      0.97      0.97      1000

